In [ ]:
import pvlib
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go


In [ ]:
### Location
lat = 55.6 #degrees North
lon = 12.1 #degrees East
albedo = 0.2

### Dates
start = pd.Timestamp('2023-01-01 00:00:00', tz='Europe/Copenhagen')
end = pd.Timestamp('2023-12-31 23:59:00', tz='Europe/Copenhagen')

### Module parameters
module_width = 1
module_length = 1.6

### System parameters
pitch = 11.75 #estimated from other measurements
length = 3.3 #estimated from other measurements
hub_height = 1.95
gcr = 0.28
max_angle = 60

In [ ]:
# create location variable
location = pvlib.location.Location(latitude=lat, longitude=lon)

In [ ]:
# get irradiance data for location
# implement some kind of local cache that saves this data ??

data, meta = pvlib.iotools.get_cams(lat, lon,start,end,'arule@dtu.dk', identifier='cams_radiation',time_step='1min')

# data, meta = pvlib.iotools.get_pvgis_tmy(lat, lon)
# data.index = data.index + pd.Timedelta(minutes=30)

data

In [9]:
# get solar position and module tilt angle
solar_position = location.get_solarposition(data.index)
module_tilt = pvlib.tracking.singleaxis(solar_position['zenith'], solar_position['azimuth'],backtrack=True, gcr=gcr, max_angle=max_angle,axis_azimuth=180)

module_tilt

,tracker_theta,aoi,surface_azimuth,surface_tilt
2023-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN
2023-01-01 00:01:00+00:00,NaN,NaN,NaN,NaN
2023-01-01 00:02:00+00:00,NaN,NaN,NaN,NaN
2023-01-01 00:03:00+00:00,NaN,NaN,NaN,NaN
2023-01-01 00:04:00+00:00,NaN,NaN,NaN,NaN
...,...,...,...,...
2023-12-31 23:55:00+00:00,NaN,NaN,NaN,NaN
2023-12-31 23:56:00+00:00,NaN,NaN,NaN,NaN
2023-12-31 23:57:00+00:00,NaN,NaN,NaN,NaN
2023-12-31 23:58:00+00:00,NaN,NaN,NaN,NaN


In [ ]:

fig = go.Figure()
for column in module_tilt.columns:
    fig.add_trace(go.Scatter(
        x=module_tilt.index,
        y=module_tilt[column],
        mode='lines',
        name=column
    ))

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='degrees',
    hovermode='x unified',
    template='plotly_white'
)
fig.show()

In [ ]:
# Get surface irradiance on module

# poa_irradiance = pvlib.bifacial.infinite_sheds.get_irradiance(
#     module_tilt['surface_tilt'], module_tilt['surface_azimuth'], 
#     solar_position['zenith'], solar_position['azimuth'], 
#     gcr, hub_height, pitch, 
#     ghi=data['ghi'], dhi=data['dhi'], dni=data['dni'], 
#     albedo=albedo, 
#     model='isotropic',
#     npoints=20,
#     vectorize=False)


model_output = pvlib.bifacial.ants2d.get_irradiance(
    tracker_rotation=module_tilt['tracker_theta'], 
    axis_azimuth=180, 
    solar_zenith=solar_position['zenith'], 
    solar_azimuth=solar_position['azimuth'], 
    gcr=gcr, 
    height=hub_height, 
    pitch=pitch, 
    ghi=data['ghi'], dhi=data['dhi'], dni=data['dni'],
    albedo=albedo,
    model='perez-driesse',)

In [10]:
model_output

,poa_front,poa_front_direct,poa_front_diffuse,poa_front_sky_diffuse,poa_front_ground_diffuse,shaded_fraction_front,poa_back,poa_back_direct,poa_back_diffuse,poa_back_sky_diffuse,poa_back_ground_diffuse,shaded_fraction_back
2023-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 00:01:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 00:02:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 00:03:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 00:04:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 23:55:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 23:56:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 23:57:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 23:58:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
fig = go.Figure()
# for column in poa_irradiance.columns:
selected_columns = ['poa_front','poa_front_direct', 'poa_front_diffuse', 'poa_back', 'poa_back_direct', 'poa_back_diffuse']
for column in selected_columns:
    fig.add_trace(go.Scatter(
        x=model_output.index,
        y=model_output[column],
        mode='lines',
        name=column
    ))

fig.update_layout(
    title='Plane-of-array irradiance',
    xaxis_title='Date',
    yaxis_title='Irradiance (W/m²)',
    hovermode='x unified',
    template='plotly_white'
)
fig.show()